# β-VAE Training on FFHQ 128×128

**Архитектура:** Convolutional β-VAE с Residual блоками и Self-Attention на 8×8  
**Loss:** `Recon (MSE) + β · KL Divergence`  
**Датасет:** FFHQ (Flickr-Faces-HQ) thumbnails 128×128  

---

## ⚙️ Конфигурация

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ── Параметры ─────────────────────────────────────────────────────────
DATA_ROOT    = '../data'          # путь к папке data/ (содержит thumbnails128x128/)
OUTPUT_DIR   = '../checkpoints/vae'

LATENT_DIM   = 256
BETA         = 4.0               # β: 1=VAE, >1=β-VAE

EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP    = 1.0

SAVE_EVERY   = 5
SAMPLE_EVERY = 5
LOG_EVERY    = 20                # логировать каждые N батчей
N_SAMPLES    = 64

DRY_RUN      = False             # True → 2 мини-эпохи без реальных данных
RESUME       = None              # путь к чекпоинту или None

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📂 Загрузка данных

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

if DRY_RUN:
    class _Dummy:
        def __init__(self): self.n = 10
        def __len__(self): return self.n
        def __iter__(self):
            for _ in range(self.n): yield torch.randn(BATCH_SIZE, 3, 128, 128)
    train_loader = val_loader = _Dummy()
    print('⚠️  DRY RUN — синтетические данные')
else:
    from data.dataset import get_dataloaders
    train_loader, val_loader = get_dataloaders(
        data_root=DATA_ROOT, batch_size=BATCH_SIZE,
        num_workers=4, val_frac=0.05,
    )

print(f'Train батчей: {len(train_loader)}')
print(f'Val батчей:   {len(val_loader)}')

# ── Превью реальных данных ────────────────────────────────────────────
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                     'text.color': '#c9d1d9', 'axes.titlecolor': '#c9d1d9'})

from torchvision.utils import make_grid
sample_batch = next(iter(train_loader))[:16]
grid = make_grid((sample_batch + 1) / 2, nrow=8, padding=2, pad_value=0.1)

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title('FFHQ — Real Images (preview)', fontsize=11)
plt.tight_layout()
plt.show()

## 🏗️ Модель

In [ ]:
from VAE.vae import VAE, vae_loss
from utils.utils import print_model_info

model = VAE(latent_dim=LATENT_DIM, beta=BETA).to(DEVICE)
print_model_info(model, f'β-VAE  (latent_dim={LATENT_DIM}, β={BETA})')

# Тест forward pass
with torch.no_grad():
    x_test = torch.randn(2, 3, 128, 128, device=DEVICE)
    recon_test, mu_test, logvar_test = model(x_test)
    total, recon_l, kl_l = vae_loss(recon_test, x_test, mu_test, logvar_test, beta=BETA)
print(f'✓ Forward pass OK: recon={recon_test.shape}, loss={total.item():.4f}')

## 🚀 Обучение

In [ ]:
import os
from pathlib import Path
from collections import defaultdict
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

from utils.utils import (
    TrainingLogger, Visualizer,
    save_checkpoint, load_checkpoint,
    save_sample_grid,
)

torch.manual_seed(42)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup    = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=2)
cosine    = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - 2), eta_min=LR * 0.01)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[2])

logger = TrainingLogger(OUTPUT_DIR, model_name='VAE')
vis    = Visualizer(OUTPUT_DIR, model_name='β-VAE', inline=True)

start_epoch = 1
best_val    = float('inf')

if RESUME and os.path.exists(RESUME):
    ckpt = load_checkpoint(RESUME, device=str(DEVICE))
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt.get('epoch', 0) + 1
    best_val    = ckpt.get('best_val', float('inf'))

_epochs = 2 if DRY_RUN else EPOCHS

for epoch in range(start_epoch, _epochs + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    t_total = t_recon = t_kl = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:3d}/{_epochs}', leave=False)
    for step, batch in enumerate(pbar):
        imgs = batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        recon, mu, logvar = model(imgs)
        total, recon_l, kl_l = vae_loss(recon, imgs, mu, logvar, beta=model.beta)
        total.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        t_total += total.item()
        t_recon += recon_l.item()
        t_kl    += kl_l.item()
        pbar.set_postfix(total=f'{total.item():.4f}',
                         recon=f'{recon_l.item():.4f}',
                         kl=f'{kl_l.item():.5f}')

    n = len(train_loader)
    train_metrics = {'total_loss': t_total/n, 'recon_loss': t_recon/n, 'kl_loss': t_kl/n}

    # ── Val ────────────────────────────────────────────────────────────
    model.eval()
    v_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch.to(DEVICE)
            recon, mu, logvar = model(imgs)
            t, _, _ = vae_loss(recon, imgs, mu, logvar, beta=model.beta)
            v_total += t.item()
    val_loss = v_total / max(len(val_loader), 1)

    scheduler.step()
    logger.log_epoch(epoch=epoch, **train_metrics, val_total=val_loss)

    is_best = val_loss < best_val
    if is_best: best_val = val_loss

    print(f'Epoch {epoch:3d}/{_epochs}  |  '
          f'train={train_metrics["total_loss"]:.4f}  '
          f'recon={train_metrics["recon_loss"]:.4f}  '
          f'kl={train_metrics["kl_loss"]:.5f}  '
          f'val={val_loss:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}'
          + ('  ★ best' if is_best else ''))

    # ── Чекпоинт ───────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        save_checkpoint(
            {'epoch': epoch, 'model': model.state_dict(),
             'optimizer': optimizer.state_dict(), 'best_val': best_val},
            OUTPUT_DIR, filename=f'checkpoint_ep{epoch:03d}.pt', is_best=is_best,
        )

    # ── Образцы + реконструкции ────────────────────────────────────────
    if epoch % SAMPLE_EVERY == 0 or epoch == _epochs:
        model.eval()
        with torch.no_grad():
            samples = model.sample(n=N_SAMPLES, device=str(DEVICE))
            save_sample_grid(samples,
                path=f'{OUTPUT_DIR}/samples/gen_ep{epoch:03d}.png',
                nrow=8, title=f'β-VAE Generated — Epoch {epoch}')

            val_imgs = next(iter(val_loader))[:8].to(DEVICE)
            recon_v, _, _ = model(val_imgs)
            comparison = torch.cat([val_imgs[:8], recon_v[:8]])
            save_sample_grid(comparison,
                path=f'{OUTPUT_DIR}/samples/recon_ep{epoch:03d}.png',
                nrow=8, title=f'β-VAE Recon — Epoch {epoch}')
        model.train()

    # ── Live plot ──────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        vis.plot_curves(logger.epoch_history, epoch=epoch, save=True, show=True)

logger.close()
print(f'\n✅ Обучение завершено! Лучший val_loss={best_val:.4f}')

## 📊 Финальные результаты

In [ ]:
# ── Финальный постер: loss curves + образцы ───────────────────────────
model.eval()
with torch.no_grad():
    final_samples = model.sample(n=N_SAMPLES, device=str(DEVICE))

poster_path = vis.plot_final_summary(
    logger.epoch_history,
    samples=final_samples,
    extra_info=f'β={BETA}',
)

from IPython.display import Image as IPyImage
IPyImage(poster_path)

In [ ]:
# ── Сравнение оригинал / реконструкция ────────────────────────────────
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

model.eval()
with torch.no_grad():
    val_imgs = next(iter(val_loader))[:8].to(DEVICE)
    recon_v, _, _ = model(val_imgs)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'text.color': '#c9d1d9', 'axes.titlecolor': '#c9d1d9'
})

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('β-VAE: Оригинал (верх) vs Реконструкция (низ)',
             fontsize=12, color='#58a6ff')

for i in range(8):
    orig  = ((val_imgs[i].cpu()  + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    recon = ((recon_v[i].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig);  axes[0, i].axis('off')
    axes[1, i].imshow(recon); axes[1, i].axis('off')

axes[0, 0].set_ylabel('Оригинал', color='#3fb950', fontsize=10)
axes[1, 0].set_ylabel('Реконструкция', color='#f78166', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/recon_comparison.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Латентная интерполяция между двумя изображениями ─────────────────
N_STEPS = 10

model.eval()
with torch.no_grad():
    imgs_ab = next(iter(val_loader))[:2].to(DEVICE)
    mu_a, logvar_a = model.encode(imgs_ab[:1])
    mu_b, logvar_b = model.encode(imgs_ab[1:])

    alphas = torch.linspace(0, 1, N_STEPS, device=DEVICE)
    interp_imgs = []
    for alpha in alphas:
        z = (1 - alpha) * mu_a + alpha * mu_b
        interp_imgs.append(model.decode(z))

interp_tensor = torch.cat(interp_imgs, dim=0)  # (N_STEPS, 3, 128, 128)

grid = make_grid((interp_tensor.cpu() + 1) / 2, nrow=N_STEPS, padding=2, pad_value=0.1)

fig, ax = plt.subplots(figsize=(20, 3))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title(f'β-VAE: Латентная интерполяция ({N_STEPS} шагов)',
             fontsize=11, color='#d2a8ff')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/interpolation.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()